In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import math
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [2]:
#clean data 
X_train = pd.read_csv('../data/processed/X_train_clean.csv')
X_val = pd.read_csv('../data/processed/X_val_clean.csv')
y_train = pd.read_csv('../data/processed/y_train_clean.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

# Combine train and val
X_full = pd.concat([X_train, X_val], ignore_index=True)
y_full = pd.concat([y_train, y_val], ignore_index=True)

print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")
print(y_full.value_counts())

X_full shape: (2403619, 71)
y_full shape: (2403619,)
Label
BENIGN                        1930547
DoS Hulk                       195605
PortScan                       134983
DDoS                           108821
DoS GoldenEye                    8749
FTP-Patator                      6745
SSH-Patator                      5012
DoS slowloris                    4927
DoS Slowhttptest                 4674
Bot                              1663
Web Attack - Brute Force         1281
Web Attack - XSS                  554
Infiltration                       31
Web Attack - Sql Injection         18
Heartbleed                          9
Name: count, dtype: int64


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

le = LabelEncoder()
y_full_encoded = le.fit_transform(y_full)

print(f"Classes: {list(le.classes_)}")

Device: mps
Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [4]:
total_samples = len(y_full)
class_freq_original = {}
for label in le.classes_:
    count = (y_full == label).sum()
    class_freq_original[label] = count / total_samples

freq_bias = torch.zeros(len(le.classes_))
for i, label in enumerate(le.classes_):
    freq_bias[i] = math.log(1.0 / class_freq_original[label])

print("Class frequency bias:")
for i, label in enumerate(le.classes_):
    print(f"  {label}: {freq_bias[i]:.4f}")

Class frequency bias:
  BENIGN: 0.2192
  Bot: 7.2761
  DDoS: 3.0950
  DoS GoldenEye: 5.6158
  DoS Hulk: 2.5086
  DoS Slowhttptest: 6.2427
  DoS slowloris: 6.1900
  FTP-Patator: 5.8759
  Heartbleed: 12.4953
  Infiltration: 11.2585
  PortScan: 2.8796
  SSH-Patator: 6.1729
  Web Attack - Brute Force: 7.5371
  Web Attack - Sql Injection: 11.8021
  Web Attack - XSS: 8.3753


In [6]:
class IAAAttention(nn.Module):
    def __init__(self, d_model, nhead):
        super(IAAAttention, self).__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.alpha = nn.Parameter(torch.tensor(0.1))
    def forward(self, x, class_bias):
        batch_size, seq_len, _ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        bias = torch.abs(self.alpha) * class_bias.view(batch_size, 1, 1, 1)
        scores = scores + bias
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        return output

class IAATransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.3):
        super(IAATransformerLayer, self).__init__()
        self.attention = IAAAttention(d_model, nhead)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, class_bias):
        attn_output = self.attention(x, class_bias)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class IAATransformer(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, num_classes, dropout=0.3):
        super(IAATransformer, self).__init__()
        self.input_projection = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([
            IAATransformerLayer(d_model, nhead, dim_feedforward=512, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, class_bias):
        x = x.unsqueeze(2)
        x = self.input_projection(x)
        for layer in self.layers:
            x = layer(x, class_bias)
        x = x.mean(dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [7]:
from sklearn.metrics import classification_report
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_full_np = X_full.values
y_full_np = y_full_encoded

all_fold_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_np, y_full_np)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/5")
    print(f"{'='*50}")
    
    # Split data
    X_tr, X_val_fold = X_full_np[train_idx], X_full_np[val_idx]
    y_tr, y_val_fold = y_full_np[train_idx], y_full_np[val_idx]
    
    # Scale
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val_fold)
    
    # SMOTE on training fold only
    sampling_strategy = {i: 10000 for i in range(15) 
                        if np.sum(y_tr == i) < 10000}
    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
    X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr_scaled, y_tr)
    
    print(f"Training samples after SMOTE: {len(X_tr_resampled)}")
    
    # DataLoaders
    freq_bias_cpu = freq_bias.cpu()
    train_loader = DataLoader(
        NetworkFlowDataset(X_tr_resampled, y_tr_resampled, freq_bias_cpu),
        batch_size=512, shuffle=True)
    val_loader = DataLoader(
        NetworkFlowDataset(X_val_scaled, y_val_fold, freq_bias_cpu),
        batch_size=512, shuffle=False)
    
    # Fresh model for each fold
    model = IAATransformer(input_size=71, d_model=128, nhead=4, 
                           num_layers=3, num_classes=15).to(device)
    freq_bias_device = freq_bias.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Train
    for epoch in range(10):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for batch_X, batch_y, batch_bias in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            batch_bias = batch_bias.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X, batch_bias)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        print(f"Epoch {epoch+1}/10 - Loss: {running_loss/len(train_loader):.4f}, Acc: {100*correct/total:.2f}%")
    
    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_X, batch_y, batch_bias in val_loader:
            batch_X = batch_X.to(device)
            batch_bias = batch_bias.to(device)
            outputs = model(batch_X, batch_bias)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.numpy())
    
    all_preds = le.inverse_transform(all_preds)
    all_labels = le.inverse_transform(all_labels)
    
    report = classification_report(all_labels, all_preds, 
                                   output_dict=True, digits=4)
    all_fold_reports.append(report)
    print(classification_report(all_labels, all_preds, digits=4))

print("\nK-Fold Complete!")


Fold 1/5
Training samples after SMOTE: 2005964
Epoch 1/10 - Loss: 0.2598, Acc: 91.68%
Epoch 2/10 - Loss: 0.1327, Acc: 94.80%
Epoch 3/10 - Loss: 0.1149, Acc: 95.35%
Epoch 4/10 - Loss: 0.1061, Acc: 95.61%
Epoch 5/10 - Loss: 0.0994, Acc: 95.83%
Epoch 6/10 - Loss: 0.0956, Acc: 95.96%
Epoch 7/10 - Loss: 0.0927, Acc: 96.07%
Epoch 8/10 - Loss: 0.0903, Acc: 96.14%
Epoch 9/10 - Loss: 0.0879, Acc: 96.21%
Epoch 10/10 - Loss: 0.0867, Acc: 96.27%
                            precision    recall  f1-score   support

                    BENIGN     0.9642    0.9787    0.9714    386110
                       Bot     0.4551    0.6246    0.5266       333
                      DDoS     0.9789    0.9675    0.9732     21765
             DoS GoldenEye     0.6881    0.7880    0.7347      1750
                  DoS Hulk     0.9572    0.7087    0.8144     39121
          DoS Slowhttptest     0.7283    0.9529    0.8256       934
             DoS slowloris     0.8895    0.9563    0.9217       985
               F

KeyboardInterrupt: 